## Environment Initialization and Dependency Conflict Resolution
As default behaviour, Colab pre-loads the latest version of `numpy` into memory (within `sys.modules`) as soon as the runtime starts. However, the `larq` module (and its associated *Keras2/TensorFlow* stack) strictly requires a `numpy` version < 2.0** (v1.26.x).

Executing a standard `%pip install larq` triggers a `numpy` downgrade that crashes the Colab dependency resolver. Since the binaries of the previous version are already allocated in the interpreter's RAM, the system hangs and forces the user to manually restart the runtime.

By introducing the following cell at the beginning of the notebook, we avoid the manual restart of the runtime: it automates the conflict resolution without requiring any manual restart.
1. **Background Installation:** It uses low-level `subprocess` to force the `numpy` downgrade and `larq` installation.
2. **Python Cache Cleanup (`sys.modules`):** It removes references to the old `numpy` modules already loaded in memory. This forces Python to import the new compatible binaries (v1.26) from scratch on the next call, eliminating the need to restart the session.

In [ ]:
import os
import sys
import time
import subprocess

# executing background pip installation for required packages
print("Updating packages...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "larq", "numpy<2.0", "matplotlib<3.9", "terminaltables", "tqdm", "tensorflow", "-q"])
print("Libraries updated successfully.")

# forcing runtime re-boot if numpy or matplotlib still have the wrong version
try:
    import numpy
    import matplotlib
    if numpy.__version__.startswith('2') or matplotlib.__version__.startswith('3.9'):
        print("Libraries still have the wrong version in memory: rebooting runtime...")
        print(" --> Please, execute again this cell after an error message pops-up in the browser window!")
        print(" --> You can simply close the error message.")

        time.sleep(1)
        os.kill(os.getpid(), 9)
except ImportError:
    pass

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import gc
import random
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ============================== #
# KERAS 2 COMPATIBILITY FOR LARQ #
# ============================== #
USE_KERAS_VERSION = 2
if USE_KERAS_VERSION == 2:
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
else:
    os.environ.pop("TF_USE_LEGACY_KERAS", None)

# ============================ #
# TENSORFLOW/KERAS/LARQ IMPORT #
# ============================ #
import tensorflow as tf
from tensorflow.keras.optimizers import SGD                                                                             # type: ignore
from tensorflow.keras.regularizers import l2                                                                            # type: ignore
from tensorflow.keras.models import Sequential                                                                          # type: ignore
from tensorflow.keras.layers import Input, AveragePooling2D, Dropout, Flatten, ReLU, Activation, BatchNormalization     # type: ignore
from tensorflow.keras.losses import SparseCategoricalCrossentropy                                                       # type: ignore
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping                                # type: ignore

from tqdm.keras import TqdmCallback

import larq as lq

from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import QuantileTransformer, MinMaxScaler

# ===================== #
# REPRODUCIBILITY SEEDS #
# ===================== #
SEEDS = [1, 42, 1234, 1809, 2602]

def initialize_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)

tf.config.experimental.enable_op_determinism()
os.environ['PYTHONHASHSEED'] = '0'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# ==================== #
# DEVICE CONFIGURATION #
# ==================== #
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  print(f"--> Using GPU: {gpus[0].name}")
else:
  print(f"--> Using CPU. No GPU detected.")

# =============== #
# DIRECTORY SETUP #
# =============== #
platforms = ['/content/drive/MyDrive/SEAI_Project', './']    # [colab, windows]
scedulers = ['adaptive_lr', 'lr_1e-1', 'lr_1e-2', 'lr_1e-3', 'lr_1e-4', 'lr_1e-5', 'lr_1e-6']
model_levels = ['baseline', 'constrained', 'quantized_16', 'quantized_8', 'quantized_4', 'quantized_2', 'nuova_norm4', 'nuova_norm2', 'prova']
neg_samples_mode = ['clean_neg_samples', 'NOT_clean_neg_samples']

current_platform = ""
current_sceduler = ""
current_model_level = ""
current_neg_samples_mode = ""

input_dir = ""

accuracy_dir = ""
accuracy_graphs_dir = ""
loss_dir = ""
loss_graphs_dir = ""
weights_dir = ""
cm_dir = ""
cm_graphs_dir = ""
roc_dir = ""
roc_graphs_dir = ""

comparison_dir = ""

baseline_accuracy_path = ""
constrained_accuracy_path = ""
quantized_16_accuracy_path = ""
quantized_8_accuracy_path = ""
quantized_4_accuracy_path = ""

def create_paths(seed):
    global input_dir, accuracy_dir, accuracy_graphs_dir, loss_dir, loss_graphs_dir, weights_dir, cm_dir, cm_graphs_dir, roc_dir, roc_graphs_dir, comparison_dir, baseline_accuracy_path, constrained_accuracy_path, quantized_16_accuracy_path, quantized_8_accuracy_path, quantized_4_accuracy_path

    input_dir = f'{current_platform}/input_cnn/scenarios17_21_{current_neg_samples_mode}_60_20_20'

    accuracy_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/accuracy'
    accuracy_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/accuracy/graphs'
    loss_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/loss"
    loss_graphs_dir = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/loss/graphs"
    weights_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/weights'
    cm_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/cm'
    cm_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/cm/graphs'
    roc_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/roc'
    roc_graphs_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/seed_{seed}/roc/graphs'

    comparison_dir = f'{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/{current_model_level}/comparison'

    baseline_accuracy_path = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/baseline/comparison/testAccuracyRegularization.npy"
    constrained_accuracy_path = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/constrained/comparison/testAccuracy.npy"
    quantized_16_accuracy_path = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/quantized_16/comparison/testAccuracy.npy"
    quantized_8_accuracy_path = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/quantized_8/comparison/testAccuracy.npy"
    quantized_4_accuracy_path = f"{current_platform}/trained_models_{current_neg_samples_mode}/{current_sceduler}/quantized_4/comparison/testAccuracyBasic.npy"

    directories = [
        accuracy_dir,
        accuracy_graphs_dir,
        loss_dir,
        loss_graphs_dir,
        weights_dir,
        cm_dir,
        cm_graphs_dir,
        roc_dir,
        roc_graphs_dir,
        comparison_dir
    ]

    for directory in directories:
        os.makedirs(directory, exist_ok=True)

## Dataset and Dataloaders definition

In [ ]:
print("Dataloaders function definition...")
def prepare_datasets(train_path, val_path, test_path, pct=1.0):
    # --------------- #
    # DATASET LOADING #
    # --------------- #
    train_data, val_data, test_data = np.load(train_path), np.load(val_path), np.load(test_path)
    train_x, train_y = train_data['features'], train_data['labels']
    val_x, val_y = val_data['features'], val_data['labels']
    test_x, test_y = test_data['features'], test_data['labels']

    total_train_samples = len(train_x)
    total_val_samples = len(val_x)
    total_test_samples = len(test_x)

    # ----------------------- #
    # PERCENTILE SUB-SAMPLING #
    # ----------------------- #
    pct_train_end_idx = int(len(train_x) * pct)
    pct_val_end_idx = int(len(val_x) * pct)
    pct_test_end_idx = int(len(test_x) * pct)
    train_x, train_y = train_x[:pct_train_end_idx], train_y[:pct_train_end_idx]
    val_x, val_y = val_x[:pct_val_end_idx], val_y[:pct_val_end_idx]
    test_x, test_y = test_x[:pct_test_end_idx], test_y[:pct_test_end_idx]

    pct_train_samples = len(train_x)
    pct_val_samples = len(val_x)
    pct_test_samples = len(test_x)

    # --------------------- #
    # Z-SCORE NORMALIZATION #
    # --------------------- #
    mean_train = np.mean(train_x)
    std_train = np.std(train_x)

    if std_train > 0:
        train_x = (train_x - mean_train) / std_train
        val_x = (val_x - mean_train) / std_train
        test_x = (test_x - mean_train) / std_train

    # ---------------------------------------- #
    # INPUT TRASLATION (POSITIVITY CONSTRAINT) #
    # ---------------------------------------- #
    min_train = np.min(train_x)
    if min_train < 0:
        traslation_offset = abs(min_train)
        train_x += traslation_offset
        val_x += traslation_offset
        test_x += traslation_offset

    # ---------- #
    # RE-SHAPING #
    # ---------- #
    if len(train_x.shape) == 3:
        train_x = np.expand_dims(train_x, axis=-1)
        val_x = np.expand_dims(val_x, axis=-1)
        test_x = np.expand_dims(test_x, axis=-1)

    return (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples)


def get_dataset(train_path, val_path, test_path, seed, pct=1.0, batch_size=64):
    (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples) = prepare_datasets(
        train_path, val_path, test_path, pct
    )

    train_set = tf.data.Dataset.from_tensor_slices((train_x, train_y))
    train_set_batched = train_set.shuffle(buffer_size=pct_train_samples, seed=seed).batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    val_set = tf.data.Dataset.from_tensor_slices((val_x, val_y))
    val_set_batched = val_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    test_set = tf.data.Dataset.from_tensor_slices((test_x, test_y))
    test_set_batched = test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    return (train_set_batched, total_train_samples, pct_train_samples), (val_set_batched, total_val_samples, pct_val_samples), (test_set_batched, total_test_samples, pct_test_samples)

print("Successfully defined.")

## CNN Model definition

In [ ]:
print("Quantized-2 model function definition...")

@lq.utils.register_keras_custom_object
def zscore_input_quantizer(x):
    return tf.quantization.fake_quant_with_min_max_args(
        x, min=0.0, max=24.0, num_bits=2    # the maximum normalized input value we have in our dataset is in range [23.0, 24.0]
    )

def build_quantized_2_model():
    model = Sequential(name="CNN_Quantized_2")
    reg_lambda = 1e-3

    model.add(Input(shape=(16, 54, 1)))

    # DoReFa for weights (maps weights using tanh distribution to [-1, 1] and quantizes)
    dorefa_weight = lq.quantizers.DoReFa(k_bit=2, mode="weights")

    # DoReFa for activations (clips activations to [0, 1] and quantizes)
    dorefa_activation = lq.quantizers.DoReFa(k_bit=2, mode="activations")

    q_weights_kwargs = {
        "kernel_quantizer": dorefa_weight,
        "kernel_constraint": lq.constraints.WeightClip(clip_value=1.0),
        "kernel_regularizer": l2(reg_lambda)
    }

    # ------- #
    # STACK 1 #
    # ------- #
    model.add(lq.layers.QuantConv2D(filters=4, kernel_size=(3, 3), padding='same', name='conv1', **q_weights_kwargs))
    model.add(BatchNormalization(name='bn1'))
    model.add(ReLU(max_value=1.0))
    model.add(AveragePooling2D(pool_size=(1, 2), name='avgpool1'))

    model.add(lq.layers.QuantConv2D(filters=4, kernel_size=(3, 3), padding='same', name='conv2', input_quantizer=dorefa_activation, **q_weights_kwargs))
    model.add(BatchNormalization(name='bn2'))
    model.add(ReLU(max_value=1.0))
    model.add(AveragePooling2D(pool_size=(2, 1), name='avgpool2'))

    # ------- #
    # STACK 2 #
    # ------- #
    model.add(lq.layers.QuantConv2D(filters=8, kernel_size=(3, 3), padding='same', name='conv3', input_quantizer=dorefa_activation, **q_weights_kwargs))
    model.add(BatchNormalization(name='bn3'))
    model.add(ReLU(max_value=1.0))
    model.add(AveragePooling2D(pool_size=(1, 3), name='avgpool3'))

    model.add(lq.layers.QuantConv2D(filters=16, kernel_size=(3, 3), padding='same', name='conv4', input_quantizer=dorefa_activation, **q_weights_kwargs))
    model.add(BatchNormalization(name='bn4'))
    model.add(ReLU(max_value=1.0))
    model.add(AveragePooling2D(pool_size=(2, 3), name='avgpool4'))

    # -------------------- #
    # PREDICTION COMPONENT #
    # -------------------- #
    model.add(Flatten(name='flatten'))
    model.add(Dropout(rate=0.2, name='dropout'))

    model.add(lq.layers.QuantDense(units=2, name='fc_output', input_quantizer=dorefa_activation, **q_weights_kwargs))
    model.add(BatchNormalization(name='bn_output'))
    model.add(Activation('softmax'))

    return model

print("Successfully defined.")

## Training and Testing cycles definition

In [ ]:
print("Training and testing cycles definition...")
def train_and_evaluate(weights_dir, scheduler_index, tp, seed, pct=1.0):
    global current_sceduler

    pct_formatted = f"{pct * 100:.1f}"
    lr = 1e-2 if scheduler_index == 0 else 1e-1 if scheduler_index == 1 else 1e-2 if scheduler_index == 2 else 1e-3 if scheduler_index == 3 else 1e-4 if scheduler_index == 4 else 1e-5 if scheduler_index == 5 else 1e-6
    current_sceduler = scedulers[scheduler_index]

    train_path = os.path.join(input_dir, 'train', f'train_Tp{tp}.npz')
    val_path = os.path.join(input_dir, 'val', f'val_Tp{tp}.npz')
    test_path = os.path.join(input_dir, 'test', f'test_Tp{tp}.npz')
    (train_set_batched, total_train_samples, pct_train_samples), (val_set_batched, total_val_samples, pct_val_samples), (test_set_batched, total_test_samples, pct_test_samples) = get_dataset(
        train_path, val_path, test_path, seed, pct=pct
    )

    # ------------- #
    # LOADING MODEL #
    # ------------- #
    print("------------------------ NEW MODEL ------------------------")
    print(f"Tp={tp}, Percentage={pct_formatted}%, Lr={lr}, Seed={seed}")

    model = build_quantized_2_model()

    for layer in model.layers:
        if isinstance(layer, (lq.layers.QuantConv2D, lq.layers.QuantDense)):
            new_weights = [tf.keras.initializers.HeNormal(seed=seed)(layer.weights[0].shape)]
            if len(layer.weights) > 1:
                new_weights.append(tf.keras.initializers.Zeros()(layer.weights[1].shape))
            layer.set_weights(new_weights)


    tp_dir = f"Tp_{tp}"
    weights_file = os.path.join(weights_dir, tp_dir, f'weights_Tp{tp}_pct{pct_formatted}.weights.h5')
    if os.path.exists(weights_file):
        print(f"Model already exists!")
        print(f" --> Loading model from: {weights_file}")
        model.load_weights(weights_file)

    model.compile(
        optimizer=SGD(learning_rate=lr, momentum=0.9, nesterov=True),
        loss=SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    # -------------- #
    # TRAINING MODEL #
    # -------------- #
    if not os.path.exists(weights_file):
        print(f"Training model...")
        print(f" --> Dataset statistics | Total samples: {total_train_samples + total_val_samples + total_test_samples}, Pct total samples: {pct_train_samples + pct_val_samples + pct_test_samples}, Pct train samples: {pct_train_samples}, Pct val samples: {pct_val_samples}, Pct test samples: {pct_test_samples}")
        os.makedirs(os.path.dirname(weights_file), exist_ok=True)

        # Callbacks #
        tqdm_cb = TqdmCallback(verbose=0)

        early_stopping_cb = EarlyStopping(
            monitor='val_loss',
            patience=20,
            min_delta=0.0,
            verbose=0,
            restore_best_weights=True
        )

        checkpoint_cb = ModelCheckpoint(
            filepath=os.path.join(weights_file),
            monitor='val_loss',
            save_best_only=True,
            save_weights_only=True,
            verbose=0
        )

        lr_scheduler = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=50,
            min_lr=1e-6,
            verbose=0
        )

        callbacks_list = [tqdm_cb, early_stopping_cb, checkpoint_cb]
        if current_sceduler == scedulers[0]:
            callbacks_list.append(lr_scheduler)

        # Actual training #
        history = None
        history = model.fit(
            train_set_batched,
            epochs=2000,
            validation_data=val_set_batched,
            callbacks=callbacks_list,
            verbose=0
        )

        # Plotting loss and accuracies #
        plot_loss(
            history.history['loss'],
            history.history['val_loss'],
            os.path.join(loss_dir, f"tloss_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(loss_dir, f"vloss_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(loss_graphs_dir, f"loss_Tp{tp}_pct{pct_formatted}.pdf"),
            tp,
            seed
        )
        plot_accuracy(
            history.history['accuracy'],
            history.history['val_accuracy'],
            os.path.join(accuracy_dir, f"taccuracy_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(accuracy_dir, f"vaccuracy_Tp{tp}_pct{pct_formatted}.npy"),
            os.path.join(accuracy_graphs_dir, f"accuracy_Tp{tp}_pct{pct_formatted}.pdf"),
            tp,
            seed
        )

    # ---------------- #
    # EVALUATING MODEL #
    # ---------------- #
    print("\nEvaluating model...")
    eval_results = model.evaluate(test_set_batched, verbose=0)
    accuracy = eval_results[1] * 100
    print(f" --> Test Accuracy: {accuracy:.2f}%")

    y_pred_probs = model.predict(test_set_batched)
    y_true = np.concatenate([y for x, y in test_set_batched], axis=0)

    plot_cm(y_pred_probs, y_true, tp, pct_formatted, seed)
    plot_ROC_curve(y_pred_probs, y_true, tp, pct_formatted, seed)
    print(f"-----------------------------------------------------------\n\n")

    tf.keras.backend.clear_session()
    gc.collect()

    return model, accuracy

print("Successfully defined.")

## Plotting Training and Validation Loss

In [ ]:
print("Defining loss plot function...")
def plot_loss(train_loss, val_loss, tloss_save_path, vloss_save_path, graph_save_path, tp, seed):
    np.save(tloss_save_path, train_loss)
    np.save(vloss_save_path, val_loss)

    epochs = range(1, len(train_loss) + 1)

    plt.figure(figsize=(5, 3.8))
    plt.plot(epochs, train_loss, label='Training Loss', color='blue', linewidth=0.6)
    plt.plot(epochs, val_loss, label='Validation Loss', color='red', linewidth=0.6)

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.suptitle(f'Training vs Validation Loss', fontsize=11, y=0.96, ha='center')
    plt.title(f'Quantized-2 Model - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    plt.subplots_adjust(top=0.85)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()

    plt.savefig(graph_save_path, format='pdf', bbox_inches='tight')
    plt.show()

print("Successfully defined.")

## Plotting Training and Validation Accuracies

In [ ]:
print("Defining accuracies plot function...")
def plot_accuracy(train_accuracy, val_accuracy, taccuracy_save_path, vaccuracy_save_path, graph_save_path, tp, seed):
    np.save(taccuracy_save_path, train_accuracy)
    np.save(vaccuracy_save_path, val_accuracy)

    epochs = range(1, len(train_accuracy) + 1)

    plt.figure(figsize=(5, 3.8))
    plt.plot(epochs, train_accuracy, label='Training Accuracy', color='blue', linewidth=0.6)
    plt.plot(epochs, val_accuracy, label='Validation Accuracy', color='red', linewidth=0.6)

    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.suptitle(f'Training VS. Validation Accuracy', fontsize=11, y=0.96, ha='center')
    plt.title(f'Quantized-2 Model - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    plt.subplots_adjust(top=0.85)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend()

    plt.savefig(graph_save_path, format='pdf', bbox_inches='tight')
    plt.show()

print("Successfully defined.")

## Plotting Test Confusion Matrix

In [ ]:
print("Defining confusion matrix plot function...")
def plot_cm(y_pred_probs, y_true, tp, pct_formatted, seed):
    y_pred_classes = np.argmax(y_pred_probs, axis=1)

    cm = confusion_matrix(y_true, y_pred_classes)
    np.save(os.path.join(cm_dir, f'cm_array_Tp{tp}_pct{pct_formatted}.npy'), cm)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.suptitle(f'Confusion Matrix', fontsize=11, y=0.96, ha='center')
    plt.title(f'Quantized-2 Model - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    plt.subplots_adjust(top=0.85)

    plt.savefig(os.path.join(cm_graphs_dir, f'cm_Tp{tp}_pct{pct_formatted}.pdf'), format='pdf', bbox_inches='tight')

    plt.show()
    plt.close()

print("Successfully defined.")

## Plotting Test ROC Curve

In [ ]:
print("Defining ROC Curve plot function...")
def plot_ROC_curve(y_pred_probs, y_true, tp, pct_formatted, seed):
    fpr, tpr, _ = roc_curve(y_true, y_pred_probs[:, 1])
    np.savez(os.path.join(roc_dir, f'roc_arrays_Tp{tp}_pct{pct_formatted}.npz'), fpr=fpr, tpr=tpr)

    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(5, 3.8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC area = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.suptitle(f'ROC Curve', fontsize=11, y=0.96, ha='center')
    plt.title(f'Quantized-2 Model - Tp={tp} - Seed={seed}', fontsize=9, pad=6)
    plt.subplots_adjust(top=0.85)
    plt.legend(loc="lower right")

    plt.savefig(os.path.join(roc_graphs_dir, f'roc_Tp{tp}_pct{pct_formatted}.pdf'), format='pdf', bbox_inches='tight')

    plt.show()
    plt.close()

print("Successfully defined.")

## Plotting Test Accuracy Comparison

In [ ]:
print("Defining accuracy plot function...")
paper_accuracies_100 = np.array([97.0, 86.6, 74.8, 69.3, 68.5, 65.8, 67.0, 65.7, 63.5, 63.7])

x_axis_tp = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

def plot_accuracy_vs_tp_comparison(accuracies):
    baseline_accuracies_100 = np.load(baseline_accuracy_path)
    constrained_accuracies_100 = np.load(constrained_accuracy_path)
    quantized_16_accuracies_100 = np.load(quantized_16_accuracy_path)
    quantized_8_accuracies_100 = np.load(quantized_8_accuracy_path)
    quantized_4_accuracies_100 = np.load(quantized_4_accuracy_path)

    plt.figure(figsize=(7, 4))

    plt.plot(x_axis_tp, accuracies, marker='o', color='crimson', label='Quantized-2 CNN')
    plt.plot(x_axis_tp, quantized_4_accuracies_100, marker='o', color='sienna', label='Quantized-4 CNN')
    plt.plot(x_axis_tp, quantized_8_accuracies_100, marker='o', color='teal', label='Quantized-8 CNN')
    plt.plot(x_axis_tp, quantized_16_accuracies_100, marker='o', color='darkgreen', label='Quantized-16 CNN')
    plt.plot(x_axis_tp, constrained_accuracies_100, marker='o', color='darkorange', label='Constrained CNN')
    plt.plot(x_axis_tp, baseline_accuracies_100, marker='o', color='navy', label='Baseline CNN')
    plt.plot(x_axis_tp, paper_accuracies_100, marker='x', color='gray', label='Paper CNN')

    plt.xlabel('Tp')
    plt.ylabel('Accuracy (%)')
    plt.suptitle('Test Accuracy vs Tp', x=0.5, y=0.95, fontsize=14)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))

    plt.tight_layout()

    plt.savefig(os.path.join(comparison_dir, f'testAccuracy_vs_tp.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

    np.save(os.path.join(comparison_dir, f'testAccuracy.npy'), accuracies)

print("Successfully defined.")

## Plotting Test Accuracy Distribution

In [ ]:
def plot_accuracy_distribution_error(tps, accuracy_distributions, mean_accuracies, scheduler_index, base_color='crimson'):
  print(f"Generating mean accuracy line plot with min-max error bars for scheduler {scheduler_index}...")

  rgb_color = mcolors.to_rgb(base_color)
  hsv_color = mcolors.rgb_to_hsv(rgb_color)

  hsv_color[2] = max(0.0, hsv_color[2] * 0.7)
  darker_color = mcolors.hsv_to_rgb(hsv_color)

  min_accuracies = [np.min(dist) for dist in accuracy_distributions]
  max_accuracies = [np.max(dist) for dist in accuracy_distributions]

  lower_errors = np.array(mean_accuracies) - np.array(min_accuracies)
  upper_errors = np.array(max_accuracies) - np.array(mean_accuracies)
  error_bounds = [lower_errors, upper_errors]

  y_limits = (45.0, 80.0)

  plt.figure(figsize=(8, 6))

  plt.errorbar(tps, mean_accuracies, yerr=error_bounds, fmt='-s',
               color=base_color, ecolor=darker_color, elinewidth=2, capsize=6,
               capthick=2, markersize=8, label='Accuracy Distribution', zorder=3)
  plt.title(f"Test Accuracy Distribution Quantized-2 Model vs TP", fontsize=15)
  plt.xlabel("TP", fontsize=13)
  plt.ylabel("Accuracy (%)", fontsize=13)
  plt.xticks(tps)
  plt.grid(axis='y', linestyle='--', alpha=0.7)
  plt.legend(loc='upper right')
  plt.ylim(y_limits)

  plt.tight_layout()

  plt.savefig(os.path.join(comparison_dir, f'accuracy_trend_lineplot_scheduler{scheduler_index}.pdf'), format='pdf', bbox_inches='tight')
  plt.show()

## Plotting Comparison with Model using New Normalization

In [ ]:
def load_experiment_data(file_path):
    # Load JSON file containing experiment metrics
    if not os.path.exists(file_path):
        print(f"Error: The file {file_path} does not exist.")
        return None

    with open(file_path, 'r') as file:
        return json.load(file)

def compare_quantization_models(base_dir, scheduler_index=3):
    q2_filename = f"results_quantized2_scheduler{scheduler_index}.json"
    nn_filename = f"results_prova_norm2_scheduler{scheduler_index}.json"

    q2_path = os.path.join(base_dir, q2_filename)
    nn_path = os.path.join(base_dir, nn_filename)

    q2_data = load_experiment_data(q2_path)
    nn_data = load_experiment_data(nn_path)

    if q2_data is None or nn_data is None:
        print("Comparison aborted due to missing data.")
        return

    tps = q2_data["tps"]
    q2_mean_acc = q2_data["mean_accuracies_by_tp"]
    nn_mean_acc = nn_data["mean_accuracies_by_tp"]

    plt.figure(figsize=(7, 4))


    plt.plot(tps, q2_mean_acc, marker='o', color='crimson', linewidth=2,
             label='Quantized-2 Model (Standard Norm)')


    plt.plot(tps, nn_mean_acc, marker='s', color='teal', linewidth=2,
             label='Quantized-2 Model (New Norm [0, 3])')

    plt.title(f"Test Accuracy Comparison: Quantized-2 Model vs New Norm Model)",
              fontsize=14, pad=15)
    plt.xlabel("TP", fontsize=12)
    plt.ylabel("Accuracy (%)", fontsize=12)
    plt.xticks(tps)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper right')
    plt.tight_layout()

    # Save and display the comparative graph
    output_graph = os.path.join(base_dir, "comparison_q2_vs_nuova_norm_prova.pdf")
    plt.savefig(output_graph, format='pdf', bbox_inches='tight')
    print(f"Graph successfully saved to {output_graph}")
    plt.show()

## Main execution

In [ ]:
if __name__ == "__main__":
    print("Executing experiment...")

    current_platform = platforms[0]                     # colab
    current_model_level = model_levels[5]               # quantized_2
    current_neg_samples_mode = neg_samples_mode[0]      # clean_neg_samples

    scheduler_indeces = [3]

    tps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    accuracies = []
    for scheduler_index in scheduler_indeces:
        current_sceduler = scedulers[scheduler_index]

        accuracies_by_tp = []
        mean_accuracies_by_tp = []
        accuracies_by_scheduler = []
        for tp in tps:
            accuracies_by_seed = []
            for seed in SEEDS:
                create_paths(seed)
                initialize_seed(seed)

                _, accuracy = train_and_evaluate(
                    weights_dir=weights_dir,
                    scheduler_index=scheduler_index,
                    tp=tp,
                    seed=seed,
                    pct=1.0
                )

                accuracies_by_seed.append(accuracy)

            accuracies_by_tp.append(accuracies_by_seed)
            mean_accuracy = np.mean(accuracies_by_seed)
            mean_accuracies_by_tp.append(mean_accuracy)

            mean_accuracy_by_tp = np.mean(accuracies_by_seed)
            accuracies_by_scheduler.append(mean_accuracy_by_tp)
        accuracies.append(accuracies_by_scheduler)

In [ ]:
def save_experiment_results(model_name, scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp):
  output_dir = os.path.join(current_platform, "experiment_results/")
  os.makedirs(output_dir, exist_ok=True)

  experiment_data = {
      "model_name": model_name,
      "scheduler_index": scheduler_index,
      "tps": tps,
      "accuracies_by_tp" : accuracies_by_tp,
      "mean_accuracies_by_tp": mean_accuracies_by_tp
  }

  output_file = os.path.join(output_dir, f"results_{model_name}_scheduler{scheduler_index}.json")

  with open(output_file, "w") as f:
    json.dump(experiment_data, f, indent=4)
  print(f"Data saved to  {output_file}")

In [ ]:
save_experiment_results("quantized2", scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp)

## Plot comparison graphs

In [ ]:
plot_accuracy_vs_tp_comparison(np.array(accuracies[0]))

In [ ]:
plot_accuracy_distribution_error(tps, accuracies_by_tp, mean_accuracies_by_tp, scheduler_index)

## New Normalization (Gaussian distribution in range [0,15])

In [ ]:
print("Dataloaders function definition...")
def prepare_datasets(train_path, val_path, test_path, k_bits, pct=1.0):
    # --------------- #
    # DATASET LOADING #
    # --------------- #
    train_data, val_data, test_data = np.load(train_path), np.load(val_path), np.load(test_path)
    train_x, train_y = train_data['features'], train_data['labels']
    val_x, val_y = val_data['features'], val_data['labels']
    test_x, test_y = test_data['features'], test_data['labels']

    total_train_samples = len(train_x)
    total_val_samples = len(val_x)
    total_test_samples = len(test_x)

    # ----------------------- #
    # PERCENTILE SUB-SAMPLING #
    # ----------------------- #
    pct_train_end_idx = int(len(train_x) * pct)
    pct_val_end_idx = int(len(val_x) * pct)
    pct_test_end_idx = int(len(test_x) * pct)
    train_x, train_y = train_x[:pct_train_end_idx], train_y[:pct_train_end_idx]
    val_x, val_y = val_x[:pct_val_end_idx], val_y[:pct_val_end_idx]
    test_x, test_y = test_x[:pct_test_end_idx], test_y[:pct_test_end_idx]

    pct_train_samples = len(train_x)
    pct_val_samples = len(val_x)
    pct_test_samples = len(test_x)

    # --------------------- #
    # NEW NORMALIZATION     #
    # --------------------- #
    num_discrete_states = (2 ** k_bits) - 1
    symmetry_offset = num_discrete_states / 2.0

    tensor_dtype = np.uint8 if k_bits <= 8 else np.uint16

    shape_train = train_x.shape
    shape_val = val_x.shape
    shape_test = test_x.shape

    flat_train = train_x.flatten().reshape(-1, 1)
    flat_val = val_x.flatten().reshape(-1, 1)
    flat_test = test_x.flatten().reshape(-1, 1)

    qt = QuantileTransformer(output_distribution='normal', n_quantiles=1000, random_state=seed)
    gauss_train = qt.fit_transform(flat_train)
    gauss_val = qt.transform(flat_val)
    gauss_test = qt.transform(flat_test)

    scaler = MinMaxScaler(feature_range=(-symmetry_offset, symmetry_offset))
    bounded_train = scaler.fit_transform(gauss_train)
    bounded_val = scaler.transform(gauss_val)
    bounded_test = scaler.transform(gauss_test)

    train_x = np.clip(np.round(bounded_train + symmetry_offset), 0, num_discrete_states).astype(tensor_dtype).reshape(shape_train)
    val_x = np.clip(np.round(bounded_val + symmetry_offset), 0, num_discrete_states).astype(tensor_dtype).reshape(shape_val)
    test_x = np.clip(np.round(bounded_test + symmetry_offset), 0, num_discrete_states).astype(tensor_dtype).reshape(shape_test)

    # ---------- #
    # RE-SHAPING #
    # ---------- #
    if len(train_x.shape) == 3:
        train_x = np.expand_dims(train_x, axis=-1)
        val_x = np.expand_dims(val_x, axis=-1)
        test_x = np.expand_dims(test_x, axis=-1)

    return (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples)


def get_dataset(train_path, val_path, test_path, seed, pct=1.0, batch_size=64):
    (train_x, train_y, total_train_samples, pct_train_samples), (val_x, val_y, total_val_samples, pct_val_samples), (test_x, test_y, total_test_samples, pct_test_samples) = prepare_datasets(
        train_path, val_path, test_path, 2, pct
    )

    train_set = tf.data.Dataset.from_tensor_slices((train_x, train_y))
    train_set_batched = train_set.shuffle(buffer_size=pct_train_samples, seed=seed).batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    val_set = tf.data.Dataset.from_tensor_slices((val_x, val_y))
    val_set_batched = val_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    test_set = tf.data.Dataset.from_tensor_slices((test_x, test_y))
    test_set_batched = test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)    # allowing CPU pre-fetch of batches during GPU execution of previous bathces

    plot_dataset_distribution(train_x, val_x, test_x, 2)

    return (train_set_batched, total_train_samples, pct_train_samples), (val_set_batched, total_val_samples, pct_val_samples), (test_set_batched, total_test_samples, pct_test_samples)

print("Successfully defined.")

In [ ]:
def plot_dataset_distribution(train_x, val_x, test_x, k_bits):

  num_discrete_states = (2 ** k_bits) - 1

  flat_train = train_x.flatten()
  flat_val = val_x.flatten()
  flat_test = test_x.flatten()

  plt.figure(figsize=(7, 4))

  bins = np.arange(-0.5, num_discrete_states + 1.5, 1.0)
  plt.hist(
      [flat_train, flat_val, flat_test],
      bins = bins,
      label = ['Train', 'Val', 'Test'],
      color = ['blue', 'orange', 'green'],
      rwidth=0.85,
      density=True
  )

  plt.title('Dataset Feature Distribution', fontsize=14)
  plt.xlabel('Feature Value', fontsize=12)
  plt.ylabel('Density', fontsize=12)

  plt.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Zero Threshold')

  plt.legend(loc='upper right')
  plt.grid(True, linestyle='--', alpha=0.6)

  plt.show()

## Training and Testing Model with New Normalization

In [ ]:
if __name__ == "__main__":
    print("Executing experiment...")

    current_platform = platforms[0]                     # colab
    current_model_level = model_levels[8]               # nuova_norm2
    current_neg_samples_mode = neg_samples_mode[0]      # clean_neg_samples

    scheduler_indeces = [3]

    tps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    accuracies = []
    for scheduler_index in scheduler_indeces:
        current_sceduler = scedulers[scheduler_index]

        accuracies_by_tp = []
        mean_accuracies_by_tp = []
        accuracies_by_scheduler = []
        for tp in tps:
            accuracies_by_seed = []
            for seed in SEEDS:
                create_paths(seed)
                initialize_seed(seed)

                _, accuracy = train_and_evaluate(
                    weights_dir=weights_dir,
                    scheduler_index=scheduler_index,
                    tp=tp,
                    seed=seed,
                    pct=1.0
                )

                accuracies_by_seed.append(accuracy)

            accuracies_by_tp.append(accuracies_by_seed)
            mean_accuracy = np.mean(accuracies_by_seed)
            mean_accuracies_by_tp.append(mean_accuracy)

            mean_accuracy_by_tp = np.mean(accuracies_by_seed)
            accuracies_by_scheduler.append(mean_accuracy_by_tp)
        accuracies.append(accuracies_by_scheduler)

In [ ]:
def save_experiment_results(model_name, scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp):
  output_dir = os.path.join(current_platform, "experiment_results/")
  os.makedirs(output_dir, exist_ok=True)

  experiment_data = {
      "model_name": model_name,
      "scheduler_index": scheduler_index,
      "tps": tps,
      "accuracies_by_tp" : accuracies_by_tp,
      "mean_accuracies_by_tp": mean_accuracies_by_tp
  }

  output_file = os.path.join(output_dir, f"results_{model_name}_scheduler{scheduler_index}.json")

  with open(output_file, "w") as f:
    json.dump(experiment_data, f, indent=4)
  print(f"Data saved to  {output_file}")

In [ ]:
save_experiment_results("prova_norm2", scheduler_index, tps, accuracies_by_tp, mean_accuracies_by_tp)

## Plot comparison graphs

In [ ]:
output_dir = os.path.join(current_platform, "experiment_results/")
compare_quantization_models(output_dir)